# Crypto Wallet Security Comparison & Ranking Dashboard


In [ ]:
# Install dependencies
!pip install -q requests pandas matplotlib numpy


# Crypto Wallet Security Comparison & Ranking

This notebook builds an interactive wallet security analyzer based on published security audit data, breach history, and feature analysis. It transforms curated security attributes into actionable rankings.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict

wallets_data = {
    'Wallet': ['Ledger Nano X', 'Ledger Nano S Plus', 'Trezor Model T', 'Trezor Model One', 'MetaMask', 'Exodus', 'Coinbase Wallet', 'Trust Wallet', 'MyEtherWallet', 'Electrum'],
    'Type': ['Hardware', 'Hardware', 'Hardware', 'Hardware', 'Software', 'Software', 'Software', 'Mobile', 'Web', 'Software'],
    'Audited': [True, True, True, True, False, True, True, False, False, True],
    'Years_Since_Major_Breach': [7, 7, 6, 6, 2, 5, 8, 3, 4, 6],
    'Open_Source': [False, False, True, True, True, False, False, True, True, True],
    'Multi_Sig_Support': [True, True, True, True, False, False, True, False, True, False],
    'Hardware_Integration': [True, True, True, True, False, False, False, False, True, False],
    'Supports_Major_Chains': [20, 20, 10, 10, 100, 50, 20, 100, 50, 5]
}

df = pd.DataFrame(wallets_data)
print("Wallet Data Loaded:")
print(df.head())

## Security Score Calculation

We calculate a composite security score based on:
- **Audit Status** (25 pts): Professional third-party audit
- **Breach History** (25 pts): Years since last major security incident
- **Open Source** (15 pts): Code transparency and community review
- **Multi-Sig** (15 pts): Multi-signature transaction support
- **Hardware Integration** (10 pts): Hardware wallet capability
- **Chain Support** (10 pts): Compatibility across ecosystems

In [ ]:
def calculate_security_score(row):
    score = 0
    
    if row['Audited']:
        score += 25
    
    years_breach = min(row['Years_Since_Major_Breach'] / 8 * 25, 25)
    score += years_breach
    
    if row['Open_Source']:
        score += 15
    
    if row['Multi_Sig_Support']:
        score += 15
    
    if row['Hardware_Integration']:
        score += 10
    
    chains_score = min(row['Supports_Major_Chains'] / 100 * 10, 10)
    score += chains_score
    
    return round(score, 1)

df['Security_Score'] = df.apply(calculate_security_score, axis=1)
df_ranked = df.sort_values('Security_Score', ascending=False).reset_index(drop=True)
df_ranked['Rank'] = range(1, len(df_ranked) + 1)

print("\n=== SECURITY RANKINGS ===")
print(df_ranked[['Rank', 'Wallet', 'Type', 'Security_Score']].to_string(index=False))

In [ ]:
def score_tier(score):
    if score >= 90:
        return 'Critical Security'
    elif score >= 75:
        return 'High Security'
    elif score >= 60:
        return 'Moderate Security'
    else:
        return 'Basic Security'

df_ranked['Tier'] = df_ranked['Security_Score'].apply(score_tier)

print("\n=== BY SECURITY TIER ===")
for tier in ['Critical Security', 'High Security', 'Moderate Security', 'Basic Security']:
    tier_wallets = df_ranked[df_ranked['Tier'] == tier]['Wallet'].tolist()
    if tier_wallets:
        print(f"\n{tier}:")
        for w in tier_wallets:
            score = df_ranked[df_ranked['Wallet'] == w]['Security_Score'].values[0]
            print(f"  • {w} ({score}/100)")

## Visualization: Security Score by Wallet Type

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax1 = axes[0]
colors = {'Hardware': '#3b82f6', 'Software': '#f59e0b', 'Mobile': '#10b981', 'Web': '#ef4444'}
df_ranked_sorted = df_ranked.sort_values('Security_Score')
bar_colors = [colors.get(t, '#6b7280') for t in df_ranked_sorted['Type']]
ax1.barh(df_ranked_sorted['Wallet'], df_ranked_sorted['Security_Score'], color=bar_colors)
ax1.axvline(x=75, color='green', linestyle='--', linewidth=2, label='High Security Threshold')
ax1.axvline(x=60, color='orange', linestyle='--', linewidth=2, label='Moderate Threshold')
ax1.set_xlabel('Security Score (0-100)', fontsize=12, fontweight='bold')
ax1.set_title('Wallet Security Rankings', fontsize=13, fontweight='bold')
ax1.legend()
ax1.grid(axis='x', alpha=0.3)

ax2 = axes[1]
type_scores = df_ranked.groupby('Type')['Security_Score'].mean().sort_values(ascending=False)
type_colors = [colors.get(t, '#6b7280') for t in type_scores.index]
ax2.bar(type_scores.index, type_scores.values, color=type_colors, edgecolor='black', linewidth=1.5)
ax2.set_ylabel('Average Security Score', fontsize=12, fontweight='bold')
ax2.set_title('Average Security by Wallet Type', fontsize=13, fontweight='bold')
ax2.set_ylim(0, 100)
for i, v in enumerate(type_scores.values):
    ax2.text(i, v + 2, f'{v:.1f}', ha='center', fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('wallet_security_ranking.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nChart saved as 'wallet_security_ranking.png'")

## Feature Comparison Matrix

In [ ]:
feature_cols = ['Audited', 'Open_Source', 'Multi_Sig_Support', 'Hardware_Integration']
feature_display = df_ranked[['Wallet', 'Type'] + feature_cols].copy()
for col in feature_cols:
    feature_display[col] = feature_display[col].map({True: '✓', False: '✗'})

print("\n=== FEATURE MATRIX ===")
print(feature_display.to_string(index=False))

print("\n=== RECOMMENDATIONS ===")
hw_secure = df_ranked[df_ranked['Type'] == 'Hardware'].iloc[0] if len(df_ranked[df_ranked['Type'] == 'Hardware']) > 0 else None
sw_secure = df_ranked[df_ranked['Type'].isin(['Software', 'Mobile'])].iloc[0] if len(df_ranked[df_ranked['Type'].isin(['Software', 'Mobile'])]) > 0 else None

if hw_secure is not None:
    print(f"\nBest Hardware Wallet: {hw_secure['Wallet']} (Score: {hw_secure['Security_Score']})")
if sw_secure is not None:
    print(f"Best Software/Mobile: {sw_secure['Wallet']} (Score: {sw_secure['Security_Score']})")

print(f"\nAverage Security Score: {df_ranked['Security_Score'].mean():.1f}")
print(f"Median Security Score: {df_ranked['Security_Score'].median():.1f}")

---

## Reference

[protraderdaily](https://protraderdaily.com/crypto/what-is-the-most-secure-crypto-wallet)
